# linkengine — quick start

A commented tour of **linkengine**: recognize, parse and normalize Italian legal citations
into stable identifiers (URN-NIR / ECLI / CELEX / PRAX). Every cell is runnable — read top to
bottom. Install the package first (`pip install -e .` from the repo root).

## 1 · Your first citation

`engine.extract(text)` returns one feature dict per recognized reference; `.rows[i]["urn"]`
is the canonical identifier.

In [1]:
from linkengine import LinkEngine

engine = LinkEngine()
result = engine.extract("art. 2697 c.c.")
print("references:", len(result.rows))
result.rows[0]["urn"]

references: 1


'urn:nir:stato:regio.decreto:1942;262:2~art2697'

## 2 · What's inside a reference

A row is a plain `dict`: the canonical `urn` plus the recognition fields (`text`, `ref-type`,
`doc-type`, `number`, `year`, `partition`, …). Empty fields just don't apply to that reference.

In [2]:
row = engine.extract("art. 43, comma 1, del d.P.R. n. 600 del 1973").rows[0]
{k: v for k, v in row.items() if v}        # show only the non-empty fields

{'ref-type': 'legislation',
 'ref-scope': 'nazionale',
 'text': 'art. 43, comma 1, del d.P.R. n. 600 del 1973',
 'context': 'art. 43, comma 1, del d.P.R. n. 600 del 1973',
 'partition': 'articolo-43_comma-1',
 'doc-type': 'DECR',
 'authority': 'PRES_REP',
 'number': '600',
 'year': '1973',
 'url': 'http://www.normattiva.it/uri-res/N2Ls?urn:nir:presidente.repubblica:decreto:1973;600~art43-comma1',
 'urn': 'urn:nir:presidente.repubblica:decreto:1973;600~art43-comma1'}

## 3 · Several citations at once (segmentation)

Real text bundles many citations; the engine splits them into one row each (and expands
ranges like `artt. 15-18`).

In [3]:
text = "Visto l'art. 2697 c.c. e la Cass. n. 100/2020, si applica il D.L. n. 34/2020."
for r in engine.extract(text).rows:
    print(f'{r["text"]:22} | {r["ref-type"]:12} | {r["urn"]}')

art. 2697 c.c          | legislation  | urn:nir:stato:regio.decreto:1942;262:2~art2697
D.L. n. 34/2020        | legislation  | urn:nir:stato:decreto.legge:2020;34
Cass. n. 100/2020      | caselaw      | ECLI:IT:CASS:2020:100CIV


## 4 · The kinds of reference

Every reference resolves to one of four identifier schemes — `ref-type` tells you which.

In [4]:
examples = {
    "legislation": "art. 19 del d.lgs. n. 546/1992",
    "caselaw":     "Cass. civ. n. 29036/2021",
    "prassi":      "Circolare AdE n. 25/E/2020",
    "EU act":      "direttiva 2006/112/CE",
}
for label, s in examples.items():
    r = engine.extract(s).rows[0]
    print(f'{label:12} | {r["ref-type"]:12} | {r["urn"]}')

legislation  | legislation  | urn:nir:stato:decreto.legislativo:1992;546~art19
caselaw      | caselaw      | ECLI:IT:CASS:2021:29036CIV
prassi       | prassi       | PRAX:AE:CIRC:2020:25/E
EU act       | legislation  | CELEX:32006L0112


## 5 · Identifier ⟷ text

`urn_to_text(urn)` is the inverse renderer — give it *only* an identifier.

In [5]:
from linkengine import urn_to_text

for u in ["ECLI:IT:CASS:2020:1234CIV",
          "urn:nir:stato:regio.decreto:1942;262:2~art2697",
          "CELEX:32016R0679",
          "PRAX:AE:CIRC:2005:47"]:
    print(f"{u:48} -> {urn_to_text(u)}")

ECLI:IT:CASS:2020:1234CIV                        -> Cassazione civile n. 1234/2020
urn:nir:stato:regio.decreto:1942;262:2~art2697   -> art. 2697 codice civile
CELEX:32016R0679                                 -> regolamento 2016/679/CE
PRAX:AE:CIRC:2005:47                             -> circolare Agenzia delle Entrate n. 47/2005


## 6 · Aliases, treaties, budget laws

Codes/consolidated texts (by name or abbreviation), bilateral tax treaties, and the annual
finanziaria / legge di bilancio all resolve to their underlying act.

In [6]:
for s in ["art. 5 c.p.c.", "art. 109 TUIR", "art. 6 GDPR",
          "Convenzione Italia-Francia, art. 15", "legge di bilancio 2023"]:
    r = engine.extract(s).rows[0]
    print(f'{s:34} -> {r["urn"]}')

art. 5 c.p.c.                      -> urn:nir:stato:regio.decreto:1940;1443:1~art5
art. 109 TUIR                      -> urn:nir:presidente.repubblica:decreto:1986;917~art109
art. 6 GDPR                        -> CELEX:32016R0679~art6
Convenzione Italia-Francia, art. 15 -> urn:nir:stato:legge:1992;20~art15
legge di bilancio 2023             -> urn:nir:stato:legge:2022;197


## 7 · Configuring context

Some references resolve only with context — pass defaults to the constructor (or to
`extract(...)` for one call).

In [7]:
# self-reference: unresolved without a court, ECLI with one
print(engine.extract("questa Corte, sent. n. 50/2019").rows[0]["urn"] or "(unresolved)")
print(LinkEngine(default_authority="CORTE_CASS")
      .extract("questa Corte, sent. n. 50/2019").rows[0]["urn"])
# regional law without a region
s = "art. 5 della legge regionale n. 4 del 2007"
print(LinkEngine(default_region="Campania").extract(s).rows[0]["urn"])

(unresolved)
ECLI:IT:CASS:2019:50CIV
urn:nir:regione.campania:legge:2007;4~art5


## 8 · Under the hood

`extract(text, debug=True)` also fills `result.trace` (the spans each recognizer found) and
`result.references` (the character offsets of each citation).

In [8]:
res = engine.extract("art. 43 del d.P.R. n. 600/1973", debug=True)
for module, spans in res.trace:
    for sp in spans:
        print(f"{module:12} {sp.entity.value:10} {sp.text!r}")

partitions   ARTICLE    'art. 43'
numbers      NUM_YEAR   'n. 600/1973'
doctypes     DOCTYPE    'd.P.R.'


## 9 · Highlight references in HTML

One function, `annotate_html`, re-emits the original text with each citation highlighted (styled
like a link). The visible text is unchanged; the fields live only in `data-*` attributes. Pick the
output with the `page` flag: `annotate_html(text)` → an inline fragment; `annotate_html(text,
page=True)` → a complete standalone document you can write to a `.html` file. Range partitions like
`artt. 15-18` anchor the inner articles to the `-`.

In [9]:
from linkengine import annotate_html
from IPython.display import HTML, display

text = "Visto l'art. 2697 c.c. e gli artt. 15-18 DPR 600/73; cfr. Cass. n. 100/2020."

# inline fragment (what you'd embed) — and the rendered, highlighted result
print(annotate_html(text))
display(HTML(annotate_html(text, page=True)))

# to save a browsable file:
#   with open("annotated.html", "w") as f:
#       f.write(annotate_html(text, page=True))

Visto l'<span class="lkn-ref" data-urn="urn:nir:stato:regio.decreto:1942;262:2~art2697" data-ref-type="legislation" data-ref-scope="nazionale" data-alias="COD_CIV" data-partition="articolo-2697" data-url="http://www.normattiva.it/uri-res/N2Ls?urn:nir:stato:regio.decreto:1942;262:2~art2697">art. 2697 c.c</span>. e gli <span class="lkn-ref" data-urn="urn:nir:presidente.repubblica:decreto:1973;600~art15" data-ref-type="legislation" data-ref-scope="nazionale" data-partition="articolo-15" data-doc-type="DECR" data-authority="PRES_REP" data-number="600" data-year="1973" data-url="http://www.normattiva.it/uri-res/N2Ls?urn:nir:presidente.repubblica:decreto:1973;600~art15">artt. 15</span><span class="lkn-ref" data-refs="2" data-urn="urn:nir:presidente.repubblica:decreto:1973;600~art16 urn:nir:presidente.repubblica:decreto:1973;600~art17" data-ref-type="legislation" data-ref-scope="nazionale" data-partition="articolo-16 articolo-17" data-doc-type="DECR" data-authority="PRES_REP" data-number="600

## 10 · Extending the library

The knowledge base is centralized: add a court in `catalog.py`, an alias in `aliases.py`, a
recognition pattern in `recognizers.py`. A quick look:

In [10]:
from linkengine import catalog, aliases

print("courts :", len(catalog.COURTS), " e.g.", list(catalog.COURTS)[:6])
print("aliases:", len(aliases.ALIASES), " e.g.", [a.code for a in aliases.ALIASES[:6]])

courts : 18  e.g. ['CORTE_CASS', 'CORTE_COST', 'CONS_STATO', 'CORTE_CONTI', 'TRIB', 'COMM_TRIBUT_REG']
aliases: 73  e.g. ['DISP_ATT_COD_PROC_CIV', 'DISP_ATT_COD_PROC_PEN', 'DISP_ATT_COD_CIV', 'COD_MIL_PACE', 'COD_MIL_GUERRA', 'COD_PROCESSO_AMM']
